# 14 - Validate Silver

Validates Delta format, key uniqueness, SCD2 interval integrity, and required MERGE evidence.

In [ ]:
import re
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
catalog = dbutils.widgets.get("catalog").strip()
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog):
    raise ValueError(f"Invalid catalog identifier: {catalog!r}")

tables = [
    f"{catalog}.silver.silver_events",
    f"{catalog}.silver.silver_orders",
    f"{catalog}.silver.silver_order_items",
    f"{catalog}.silver.silver_customers",
    f"{catalog}.silver.silver_order_status_history",
]
details = []
for table in tables:
    exists = spark.catalog.tableExists(table)
    detail = spark.sql(f"DESCRIBE DETAIL {table}").first() if exists else None
    details.append({"table_name": table, "exists": exists, "format": detail["format"] if detail else None, "num_files": detail["numFiles"] if detail else None})
details_df = spark.createDataFrame(details)
details_df.display()
if details_df.filter((~F.col("exists")) | (F.col("format") != "delta")).count():
    raise RuntimeError("Missing or non-Delta Silver table")

events = spark.table(f"{catalog}.silver.silver_events")
orders = spark.table(f"{catalog}.silver.silver_orders")
items = spark.table(f"{catalog}.silver.silver_order_items")
history = spark.table(f"{catalog}.silver.silver_order_status_history")
checks = {
    "events_duplicate_key": events.groupBy("event_id").count().filter(F.col("count") > 1).count(),
    "events_invalid_required": events.filter(F.col("event_id").isNull() | F.col("event_time").isNull()).count(),
    "events_invalid_device": events.filter(F.col("device.os").isNull() | F.col("device.model").isNull()).count(),
    "events_wrong_event_date": events.filter(F.col("event_date") != F.to_date("event_time")).count(),
    "orders_duplicate_key": orders.groupBy("order_id").count().filter(F.col("count") > 1).count(),
    "orders_not_latest_state": orders.alias("orders").join(history.groupBy("order_id").agg(F.max("status_ts").alias("max_status_ts")).alias("history"), "order_id").filter(F.col("orders.updated_at") != F.col("history.max_status_ts")).count(),
    "items_duplicate_key": items.groupBy("order_id", "sku").count().filter(F.col("count") > 1).count(),
    "items_fx_mismatch": items.filter(F.abs(F.col("unit_price_usd") - (F.col("unit_price_original") * F.col("fx_rate_to_usd"))) > F.lit(0.01)).count(),
    "customers_multiple_current": spark.table(f"{catalog}.silver.silver_customers").filter("is_current").groupBy("user_id").count().filter(F.col("count") != 1).count(),
    "customers_missing_current": spark.table(f"{catalog}.silver.silver_customers").select("user_id").distinct().join(spark.table(f"{catalog}.silver.silver_customers").filter("is_current").select("user_id").distinct(), "user_id", "left_anti").count(),
}
customers = spark.table(f"{catalog}.silver.silver_customers")
checks["customers_current_interval_mismatch"] = customers.filter((F.col("is_current") & F.col("effective_to").isNotNull()) | (~F.col("is_current") & F.col("effective_to").isNull())).count()
timeline = Window.partitionBy("user_id").orderBy("effective_from")
checks["customers_overlapping_intervals"] = customers.withColumn("_previous_to", F.lag("effective_to").over(timeline)).filter(F.col("_previous_to").isNotNull() & (F.col("effective_from") <= F.col("_previous_to"))).count()
checks_df = spark.createDataFrame([{"check_name": key, "violation_count": value} for key, value in checks.items()])
checks_df.display()
if checks_df.filter(F.col("violation_count") != 0).count():
    raise RuntimeError("Silver quality validation failed")

for table in [f"{catalog}.silver.silver_orders", f"{catalog}.silver.silver_customers"]:
    merge_count = spark.sql(f"DESCRIBE HISTORY {table}").filter(F.col("operation") == "MERGE").count()
    if merge_count == 0:
        raise RuntimeError(f"Required MERGE evidence missing for {table}")

print("Silver validation passed")